In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ.Ab8RN


In [3]:
openai = OpenAI()
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [4]:
def shout(text):
    print(f"Shout has been called with input {text} ")
    return text.upper()

In [5]:
shout("hello")

Shout has been called with input hello 


'HELLO'

In [13]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7898
* To create a public link, set `share=True` in `launch()`.


In [9]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7894
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input hello 
Shout has been called with input pramod 
Shout has been called with input pramod 


In [10]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("admin", "admin"))

* Running on local URL:  http://127.0.0.1:7895
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input pramod 


In [11]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input hello my name is Pramod. Whats up 
Shout has been called with input hello 
Shout has been called with input howdy 


In [31]:
def stream_gpt(prompt):
    system_message = "You are a helpful assistant"
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    # return stream.choices[0].messsage.content
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
# stream_gpt("tell me a joke")

AttributeError: 'Choice' object has no attribute 'messsage'

In [32]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7914
* To create a public link, set `share=True` in `launch()`.


In [33]:
def stream_gemini(prompt):
    system_message = "You are a helpful assistant"
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = gemini.chat.completions.create(
        model='gemini-3.1-flash-lite',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for Claude 4.5 Sonnet", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gemini,
    title="Gemini", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7915
* To create a public link, set `share=True` in `launch()`.


In [35]:
from scrapper import fetch_website_contents

def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "Gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    
    yield from result

In [36]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Gemini"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Tabs and Space", "https://alexkondov.com/indentation-warfare-tabs-vs-spaces/", "Gemini"]
        ], 
    flagging_mode="never"
    )
view.launch()


* Running on local URL:  http://127.0.0.1:7916
* To create a public link, set `share=True` in `launch()`.


In [37]:
SYSTEM_PROMPT = """
You are an expert AI assistant in resolving user queries using Chain of thought.
You work on START, PLAN and OUTPUT steps. 
You need to first PLAN what needs to be done. The PLAN can be multiple steps. 
Once you think enough PLAN has been done. finally you can give an OUTPUT. 

Rules:
-strictly follow the Json output format
-Only run one step at a time. 
- The sequence of steps is START(where user give an input), PLAN (that can be multiple times) and finally
OUTPUT (which is going to be displayed to the user)

OUTPUT Json Format: 
{"step": "START"| "PLAN"| "OUTPUT", "content": "<string>"}

Example:
START: Hey, Can you solve 2+3 * 5 /10
PLAN: {"step": "PLAN", "content": "Seems like user is interested to solve a math problem"}
PLAN: {"step": "PLAN", "content": "Looking at the problem, we should solve using BODMAS method"}
PLAN: {"step": "PLAN", "content": "Yes, The BODMAS is correct way of solving it"}
PLAN: {"step": "PLAN", "content": "first we my divide by 5 by 10 which is 0.5"}
PLAN: {"step": "PLAN", "content": "now we my multiply by 0.5 with 3 which is 1.5"}
PLAN: {"step": "PLAN", "content": "then we my add by 1.5 with 2 and the answer will be 3.5"}
PLAN: {"step": "PLAN", "content": "Great, we have solved and finally left with 3.5 as answer"}
OUTPUT: {"step": "OUTPUT", "content": "The answer is 3.5"}

"""

In [38]:
import json

def chat(message, history):

    message_history = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Previous conversation
    for user, assistant in history:
        message_history.append({"role": "user", "content": user})
        message_history.append({"role": "assistant", "content": assistant})

    # Current question
    message_history.append({"role": "user", "content": message})

    while True:
        response = openai.chat.completions.create(
            model="gpt-4.1-mini",
            response_format={"type": "json_object"},
            messages=message_history
        )

        raw_result = response.choices[0].message.content
        message_history.append({"role": "assistant", "content": raw_result})

        parsed_result = json.loads(raw_result)

        if parsed_result["step"] == "START":
            yield f"🔥 {parsed_result['content']}"

        elif parsed_result["step"] == "PLAN":
            yield f"🧠 {parsed_result['content']}"

        elif parsed_result["step"] == "OUTPUT":
            yield f"🤖 {parsed_result['content']}"
            break

In [39]:
import gradio as gr
demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7917
* To create a public link, set `share=True` in `launch()`.
